# 008 Implement a Real Open-Meteo Fallback Path

这是第八课：实现真正的 Open-Meteo fallback 路径。

学习目标：

1. 理解 fallback 从“概念”变成“真实实现”时会发生什么
2. 学会为 fallback 路径增加独立脚本，而不是把逻辑塞回主路径脚本
3. 用真实天气 Skill 实现第一个可运行的 Open-Meteo 查询构建器
4. 理解为什么 fallback 路径也应该有清晰边界

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

第七课里，我们已经讲清楚了：

- 什么情况下走主路径
- 什么情况下走 fallback

但当时 fallback 还停留在“决策层”。

现在这节课要继续推进一步：

- 真正把 Open-Meteo fallback 路径实现出来

也就是说，现在不是只说“可以切 fallback”，而是让它真的能产出一个可用查询 URL。


## 先看当前真实目录

这次我们继续扩展同一份天气 Skill。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   ├── __pycache__
│   │   ├── normalize_location.cpython-310.pyc
│   │   └── normalize_location.cpython-313.pyc
│   ├── build_open_meteo_query.py
│   ├── build_wttr_query.py
│   └── normalize_location.py
└── SKILL.md


## 先看 `SKILL.md` 的变化

这次你应该重点观察：

- `Scripts` 小节增加了 `build_open_meteo_query.py`

这意味着 fallback 已经不只是“可选思路”，而是 Skill 的一部分真实能力。


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 再看 `references/weather_sources.md` 的变化

这次 reference 也同步扩展了。

重点不是加很多内容，而是把“教学版 fallback 的边界”讲清楚：

- 这不是完整地理编码系统
- 这里先用小型 demo 坐标映射支撑教学


In [4]:
print((skill_root / 'references' / 'weather_sources.md').read_text(encoding='utf-8'))


# Weather Sources

This reference contains the concrete query patterns for the weather skill.

Use `wttr.in` as the primary source for concise, human-readable weather output.

Use Open-Meteo as the fallback when a structured JSON response is more useful.

## wttr.in

Quick one-liner:

```bash
curl -s "wttr.in/London?format=3"
# Output: London: ⛅️ +8°C
```

Compact format:

```bash
curl -s "wttr.in/London?format=%l:+%c+%t+%h+%w"
# Output: London: ⛅️ +8°C 71% ↙5km/h
```

Full forecast:

```bash
curl -s "wttr.in/London?T"
```

Format codes:

- `%c` condition
- `%t` temp
- `%h` humidity
- `%w` wind
- `%l` location
- `%m` moon

Tips:

- URL-encode spaces: `wttr.in/New+York`
- Airport codes: `wttr.in/JFK`
- Units: `?m` for metric, `?u` for USCS
- Today only: `?1`
- Current only: `?0`
- PNG output: `curl -s "wttr.in/Berlin.png" -o /tmp/weather.png`

## Open-Meteo

Use this when a structured JSON response is more useful than text.

```bash
curl -s "https://api.open-meteo.com/v1/forecast?latitu

## 为什么 fallback 也要独立成脚本

一个常见错误是：

- 把 fallback 逻辑继续堆进 `build_wttr_query.py`

这会让主路径和 fallback 路径耦合得越来越重。

更稳的做法是：

- `build_wttr_query.py` 只负责 `wttr.in`
- `build_open_meteo_query.py` 只负责 Open-Meteo

这样两条路径边界清晰，后面更容易维护。


In [5]:
path_split = {
    'primary_script': 'build_wttr_query.py',
    'fallback_script': 'build_open_meteo_query.py',
    'shared_helper': 'normalize_location.py',
}

from pprint import pprint
pprint(path_split)


{'fallback_script': 'build_open_meteo_query.py',
 'primary_script': 'build_wttr_query.py',
 'shared_helper': 'normalize_location.py'}


## 读真实 fallback 脚本

这份脚本的教学重点有两个：

1. 它复用了 `normalize_location()`
2. 它用一个小型坐标映射支撑 fallback，而没有一下子扩成完整地理编码系统

这是一种很适合教学和早期开发的做法。


In [6]:
print((skill_root / 'scripts' / 'build_open_meteo_query.py').read_text(encoding='utf-8'))


#!/usr/bin/env python3
import argparse

from normalize_location import normalize_location


DEMO_COORDS = {
    "Beijing": (39.9042, 116.4074),
    "Shanghai": (31.2304, 121.4737),
    "London": (51.5072, -0.1276),
    "New+York": (40.7128, -74.0060),
    "JFK": (40.6413, -73.7781),
}


def build_open_meteo_query(location: str) -> str:
    normalized = normalize_location(location)
    if normalized not in DEMO_COORDS:
        supported = ", ".join(sorted(DEMO_COORDS))
        raise ValueError(
            f"Unsupported demo location: {normalized}. Supported demo locations: {supported}"
        )

    latitude, longitude = DEMO_COORDS[normalized]
    return (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current_weather=true"
    )


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Build an Open-Meteo fallback URL from a user-provided location."
    )
    parser.add_argument("location", help="Raw locati

## 直接跑几个支持的地点

先验证 happy path。


In [7]:
import subprocess

supported = ['beijing', 'shanghai', ' New   York ', 'jfk']

for location in supported:
    result = subprocess.run(
        [
            'python',
            '.agents/skills/weather-query-assistant/scripts/build_open_meteo_query.py',
            location,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    print(location, '->', result.stdout.strip(), '| returncode =', result.returncode)


beijing -> https://api.open-meteo.com/v1/forecast?latitude=39.9042&longitude=116.4074&current_weather=true | returncode = 0
shanghai -> https://api.open-meteo.com/v1/forecast?latitude=31.2304&longitude=121.4737&current_weather=true | returncode = 0
 New   York  -> https://api.open-meteo.com/v1/forecast?latitude=40.7128&longitude=-74.006&current_weather=true | returncode = 0
jfk -> https://api.open-meteo.com/v1/forecast?latitude=40.6413&longitude=-73.7781&current_weather=true | returncode = 0


## 再看一个不支持的地点

这一步很关键。

fallback 不是“能兜底一切”，而是“在定义好的边界内提供替代路径”。

如果教学版 fallback 现在只支持少量 demo 地点，就应该明确报错，而不是悄悄瞎猜。


In [8]:
unsupported = subprocess.run(
    [
        'python',
        '.agents/skills/weather-query-assistant/scripts/build_open_meteo_query.py',
        'chengdu',
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(unsupported.stdout.strip())
print('returncode =', unsupported.returncode)


Unsupported demo location: Chengdu. Supported demo locations: Beijing, JFK, London, New+York, Shanghai
returncode = 2


## 这一步为什么比“直接查 API”更重要

很多初学者会想：

- 为什么不直接把完整 API 请求也做完？

原因是这节课的重点不是联网本身，而是 Skill 的 fallback 架构。

这一步最重要的成果是：

- fallback 有了真实脚本
- fallback 有了明确输入输出
- fallback 有了支持范围和失败边界

这些比“多打一条 HTTP 请求”更关键。


In [9]:
why_this_matters = [
    'fallback is now implemented, not only described',
    'input/output boundary is explicit',
    'supported scope is explicit',
    'failure path is explicit',
]

pprint(why_this_matters)


['fallback is now implemented, not only described',
 'input/output boundary is explicit',
 'supported scope is explicit',
 'failure path is explicit']


## 现在这份天气 Skill 的结构已经很完整了

到第八课为止，它已经具备：

- `SKILL.md`
- `agents/openai.yaml`
- `references/`
- 主路径脚本
- fallback 脚本
- 会话分支教学

也就是说，Skill 的骨架已经接近一个完整示例了。


In [10]:
weather_skill_state = {
    'workflow': True,
    'metadata': True,
    'references': True,
    'primary_script': True,
    'fallback_script': True,
    'session_examples': True,
}

pprint(weather_skill_state)


{'fallback_script': True,
 'metadata': True,
 'primary_script': True,
 'references': True,
 'session_examples': True,
 'workflow': True}


## 如果继续往下走，第九课最自然的方向是什么

现在最值得补的，不是再继续加功能，而是开始讲“验证”。

所以第九课最自然的方向是：

- 如何验证这份 Skill 是否真的可用

例如可以讲：

1. 怎么验证脚本输出是否符合预期
2. 怎么验证主路径和 fallback 路径都能走通
3. 怎么判断 Skill 说明是否足够清楚


In [11]:
lesson_nine_candidate = {
    'theme': 'validation and testing for the weather skill',
    'topics': [
        'script output checks',
        'primary vs fallback path checks',
        'skill instruction clarity checks',
    ],
}

pprint(lesson_nine_candidate)


{'theme': 'validation and testing for the weather skill',
 'topics': ['script output checks',
            'primary vs fallback path checks',
            'skill instruction clarity checks']}


## 当前阶段结论

你现在需要记住：

1. 真正的 fallback 实现，应该有独立边界，而不是继续塞进主路径脚本
2. `build_open_meteo_query.py` 让这份天气 Skill 第一次拥有了真实可运行的 fallback 能力
3. 教学版 fallback 可以先用小型 demo 映射，不必一开始就做成完整地理编码系统
4. 一份 Skill 的 fallback 路径，最好同样具备明确的输入、输出、支持范围和错误边界
5. 如果按当前路线继续，下一课最自然就是讲验证和测试

下一步建议：

- 继续第九课：验证这份天气 Skill 是否真的可用
